## load_fhfa
Loads the FHFA **HPI master** CSV (`hpi_master_all_geographies.csv`, all geographies / frequencies / methodology variants in one long file) from `RAW_FHFA` (`/Volumes/{CATALOG}/raw/fhfa/`) into the all-STRING Bronze table `{BRONZE}.fhfa_hpi_master`. The two quarterly xlsx are verified subsets of master and are not ingested.

**Write strategy (A):** MERGE on the full grain `(hpi_type, hpi_flavor, frequency, level, place_id, yr, period)` — the master file mixes monthly + quarterly and multiple flavors per place, so a shorter key would collapse distinct rows. Re-running the same file is a no-op; a revised release updates in place.

**No archiving (deliberate deviation from CLAUDE.md §10 cell 7, recorded per §18):** FHFA is a rolling full snapshot re-downloaded each cycle by the idempotent download framework. Archiving the raw CSV would serve no purpose here. DDL: `libs/ddl/bronze_ddl.py`.

In [ ]:
%run "../libs/notebook_init"

In [ ]:
# notebook_init injected: BRONZE, AUDIT, RAW_FHFA, PIPELINE_RUN_ID, STATUS_*, StepLog,
# Utils, ingestion_log_insert, spark, dbutils, F, StructType/StructField/StringType.

STEP_SEQUENCE = 1                                   # position is owned by the orchestrator
SOURCE_SYSTEM = "fhfa"
SOURCE_PATH   = RAW_FHFA                            # /Volumes/{CATALOG}/raw/fhfa/
TARGET_TABLE  = f"{BRONZE}.fhfa_hpi_master"

# Exact source header (12 cols), in file order. Single source of truth: the read schema
# is derived from it, and cell 4 validates each file's header against it (fail on drift).
EXPECTED_SOURCE_COLS = [
    "hpi_type", "hpi_flavor", "frequency", "level", "place_name", "place_id",
    "yr", "period", "index_nsa", "index_sa", "rstderr", "note",
]
# Bronze = all STRING (source fidelity; cast at Silver). FHFA columns are already
# snake_case, so no rename is needed downstream.
read_schema = StructType([StructField(col_name, StringType(), True) for col_name in EXPECTED_SOURCE_COLS])

# MERGE natural key — the full grain (see strategy note above).
MERGE_KEYS = ["hpi_type", "hpi_flavor", "frequency", "level", "place_id", "yr", "period"]

In [ ]:
# Open the pipeline_step_log row (RUNNING). Closed explicitly in cell 6 (succeed) or by
# step.fail(e) in any work cell's 2-line handler.
nb = Utils.get_notebook_context(dbutils)
step = StepLog(
    spark, AUDIT, dbutils,
    pipeline_run_id = PIPELINE_RUN_ID,
    step_sequence   = STEP_SEQUENCE,
    notebook_folder = nb["notebook_folder"],
    notebook_name   = nb["notebook_name"],
    layer           = "bronze",
    target_table    = TARGET_TABLE,
)
print(f"load_fhfa: step_log_id={step.step_log_id}")

In [ ]:
# Per-file header validation BEFORE the bulk read (Spark's directory read merges files
# and cannot detect per-file drift). The no-files CHECK stays inside the try (a failed
# dbutils.fs.ls is logged via step.fail); the early EXIT goes OUTSIDE the try, because
# dbutils.notebook.exit() raises an ordinary exception that `except Exception` would
# swallow (no dbutils.NotebookExit class) — see CLAUDE.md §10.1 / gotchas.
no_files = False
try:
    files = [file_info.path for file_info in dbutils.fs.ls(SOURCE_PATH) if file_info.path.lower().endswith(".csv")]
    no_files = not files
    if not no_files:
        bad_files = []
        for file_path in files:
            actual = (
                spark.read.format("csv").option("header", "true")
                .load(file_path).limit(0).columns
            )
            if actual != EXPECTED_SOURCE_COLS:
                bad_files.append((file_path, actual))
        if bad_files:
            raise ValueError(
                f"[{TARGET_TABLE}] Header mismatch in {len(bad_files)} file(s).\n"
                f"Expected: {EXPECTED_SOURCE_COLS}\n"
                + "\n".join(f"  {file_path}\n    actual: {actual_header}" for file_path, actual_header in bad_files)
            )
        print(f"load_fhfa: {len(files)} file(s) passed header validation.")
except Exception as e:
    step.fail(e); raise

if no_files:
    step.no_files()
    dbutils.notebook.exit(f"No CSV files found at {SOURCE_PATH}")

In [ ]:
# Explicit schema (never inferSchema). Add audit columns: source_file_path from file
# metadata, inserted_ts via current_timestamp (executor-consistent, not datetime.now),
# run_id from the pipeline run. Column names already match the DDL (snake_case source).
try:
    raw_df = (
        spark.read
            .format("csv")
            .option("header", "true")
            .option("delimiter", ",")
            .schema(read_schema)
            .load(SOURCE_PATH)
            .withColumn("source_file_path", F.col("_metadata.file_path"))
    )
    shaped_df = (
        raw_df
            .withColumn("inserted_ts", F.current_timestamp())
            .withColumn("run_id", F.lit(PIPELINE_RUN_ID))
    )
    rows_read = shaped_df.count()
    step.rows_read = rows_read
    print(f"load_fhfa: read {rows_read:,} rows from {SOURCE_PATH}")
except Exception as e:
    step.fail(e); raise

In [ ]:
# Strategy A: MERGE on the natural key. INSERT */UPDATE SET * match by column name (the
# staging columns are exactly the target columns). MERGE metrics (num_inserted_rows /
# num_updated_rows) come back as the result row on Databricks [Projected — confirm on
# first run]; fall back to the post-pre delta for the insert count if absent.
try:
    shaped_df.createOrReplaceTempView("fhfa_staging")
    on_clause = " AND ".join(f"t.{key_col} = s.{key_col}" for key_col in MERGE_KEYS)

    pre_count = spark.table(TARGET_TABLE).count()
    metrics = spark.sql(f"""
        MERGE INTO {TARGET_TABLE} AS t
        USING fhfa_staging AS s
        ON {on_clause}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """).first().asDict()
    post_count = spark.table(TARGET_TABLE).count()

    inserted = metrics.get("num_inserted_rows")
    updated  = metrics.get("num_updated_rows")
    if inserted is None:
        inserted = post_count - pre_count
    if post_count - pre_count != inserted:
        raise AssertionError(
            f"[{TARGET_TABLE}] Insert-count mismatch: MERGE reported {inserted:,} inserts, "
            f"but row count grew by {post_count - pre_count:,} "
            f"(pre {pre_count:,}, post {post_count:,})."
        )

    step.rows_written = inserted
    step.succeed()
    print(f"load_fhfa: MERGE done — inserted={inserted:,} updated={updated} "
          f"(read={rows_read:,}, pre={pre_count:,}, post={post_count:,}).")
except Exception as e:
    step.fail(e); raise

# ingestion_log (leaf tier) runs AFTER succeed() and OUTSIDE the write try: the helper
# swallows its own errors and returns a dict, so a logging hiccup can't roll back the
# committed MERGE (CLAUDE.md §11.4/§12). One row per ingested file.
files_df = shaped_df.select("source_file_path").distinct()
res = ingestion_log_insert(
    spark, AUDIT, files_df, PIPELINE_RUN_ID, step.step_log_id,
    source_system=SOURCE_SYSTEM, target_table=TARGET_TABLE,
)
if res["status"] != STATUS_SUCCEEDED:
    print(f"load_fhfa: WARNING ingestion_log insert failed: {res['error_message']}")